# 09 · Panel municipal: análisis de los municipios rurales

> **En BigQuery.** Lee `DEFORESTACION_CREDITO_PRODUCCION_MUNICIPIO_ANIO` y `AREA_COSECHADA_EVA_MUNICIPIO_CULTIVO_ANIO` de `DATASET_DEFORESTACION`. Para el mapa, descarga del DANE los límites municipales (no los guarda). No escribe nada. **Qué tienes que cambiar:** nada; la celda 1 ya apunta al proyecto del Observatorio (`prueba-ofr`). **Cuánto tarda:** un par de minutos.
>
> Los años son los del trabajo de grado (2021 a 2025, y 2021 a 2024 para la EVA). Si el panel municipal se actualiza con años nuevos, este notebook sigue mostrando esos.

Este notebook analiza el panel que arma el notebook 06 (`DEFORESTACION_CREDITO_PRODUCCION_MUNICIPIO_ANIO` y `AREA_COSECHADA_EVA_MUNICIPIO_CULTIVO_ANIO`). Trabaja con dos filtros:

- **Municipios**: solo los que la clasificación del DNP pone como **rurales** o **rurales dispersos**. La excepción es
  el mapa de la sección 5, que muestra todos los municipios del país.
- **Crédito**: solo el crédito de FINAGRO que va al **eslabón de producción** de la cadena
  (`eslabon_cadena = Producción`), de todos los tipos de productor.

Con eso hace cinco cosas:

1. Un **índice** de cómo se movieron la deforestación, el crédito, el ganado y los cultivos.
2. **Curvas de Lorenz** y gráficos de **Pareto 80-20**: ¿en cuántos municipios se concentra cada cosa?
3. Los **cultivos de los 10 municipios que más deforestan**.
4. **Correlaciones** entre la deforestación y el crédito, el ganado y los cultivos.
5. Un **mapa** de la deforestación de todos los municipios.

Todo lo que sigue son **asociaciones**: dice qué cosas aparecen juntas, no qué causa qué.

## Cómo leer este notebook

Cada sección tiene tres partes:

1. **La pregunta** que intenta responder.
2. **Cómo se hace**, en palabras.
3. **Cómo leer el gráfico.**

In [ ]:
# =====================================================================
# 1. CONFIGURACIÓN
#    Ya está llena para el proyecto del Observatorio (prueba-ofr). Solo
#    hay que cambiarla si las tablas se pasan a otro proyecto o dataset.
# =====================================================================
# --- Dónde están las tablas en BigQuery ------------------------------
# Proyecto de Google Cloud del Observatorio (Prueba-OFR). Es el id,
# no el nombre: el que aparece en la consola al elegir el proyecto.
PROYECTO = "prueba-ofr"

# Dataset con las tablas de deforestación que arman los notebooks 01
# a 06 (grilla, paneles y panel municipal).
DATASET_DEFORESTACION = "Cambio_Climatico"

# --- Solo si corres el notebook en tu computador ----------------------
# En BigQuery déjalo vacío. En un computador: la ruta al archivo .json
# de la llave de una cuenta de servicio con acceso a estas tablas.
RUTA_LLAVE_BIGQUERY = ""

In [ ]:
# =====================================================================
# 2. INSTALAR PAQUETES
#    En BigQuery instala lo que le falte a la máquina; la primera vez
#    tarda uno o dos minutos. No hay que cambiar nada. En un computador
#    con el entorno del repositorio (requirements_local.txt) se puede
#    saltar.
#    Las versiones fijas son las que se usaron en el trabajo de grado,
#    para que los resultados salgan iguales.
# =====================================================================
%pip install --quiet geopandas==1.1.1 seaborn pandas-gbq

In [ ]:
# =====================================================================
# 3. CONEXIÓN CON BIGQUERY Y FUNCIONES DE APOYO
#    No hay que cambiar nada en esta celda. Se conecta a BigQuery y
#    define las funciones con que el resto del notebook lee y escribe
#    tablas. Es igual en todos los notebooks de esta carpeta.
# =====================================================================
import json
import re
import tempfile
import unicodedata
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
from google.cloud import bigquery

NOMBRE_NOTEBOOK = "09_eda_panel_municipal"

# --- 3.1 ¿Quedó algo sin llenar en la celda 1? --------------------------
# Si queda algún <<LLENAR ...>>, el notebook se detiene aquí y dice cuál.
_pendientes = sorted(n for n, v in list(globals().items())
                     if n.isupper() and isinstance(v, str) and "<<LLENAR" in v)
if _pendientes:
    raise ValueError("Falta llenar en la celda 1: " + ", ".join(_pendientes))

# --- 3.2 Conexión ---------------------------------------------------------
# En BigQuery el notebook entra con la cuenta de quien lo corre y no
# necesita llave. La llave solo se usa si se corre en un computador.
if globals().get("RUTA_LLAVE_BIGQUERY"):
    from google.oauth2 import service_account
    client = bigquery.Client(
        project=PROYECTO,
        credentials=service_account.Credentials.from_service_account_file(RUTA_LLAVE_BIGQUERY))
else:
    client = bigquery.Client(project=PROYECTO)
print(f"Conectado a BigQuery: proyecto {PROYECTO}")

# Carpeta para los archivos que se descargan mientras corre el notebook.
# Es de paso: todo lo que importa termina en tablas de BigQuery.
_carpeta = globals().get("CARPETA_DESCARGAS") or ""
DIR_TRABAJO = Path(_carpeta) if _carpeta else Path(tempfile.gettempdir()) / "ofr_deforestacion"
DIR_TRABAJO.mkdir(parents=True, exist_ok=True)


# --- 3.3 Nombres de tabla -----------------------------------------------
def nombre_completo(tabla, dataset=None):
    """'deforestacion_ideam_celda_5km_periodo' -> 'proyecto.dataset.DEFORESTACION_IDEAM_CELDA_5KM_PERIODO'.

    Las tablas que crean estos notebooks se nombran en mayúscula
    (GRILLA_CELDAS_5KM, DEFORESTACION_IDEAM_CELDA_5KM_PERIODO...). Si la tabla ya viene completa
    (proyecto.dataset.tabla, como las del Observatorio), se devuelve igual.
    Si no, se arma con PROYECTO y el dataset indicado; por defecto,
    DATASET_DEFORESTACION.
    """
    if tabla.count(".") == 2:
        return tabla
    return f"{PROYECTO}.{dataset or globals().get('DATASET_DEFORESTACION')}.{tabla.upper()}"


# --- 3.4 Leer -------------------------------------------------------------
_MARCA = re.compile(r"^(\[índice(?: sin nombre)?\] )?(?:Nombre original: «(.*?)»)?")


_LECTURA_RAPIDA = [True]


def leer_sql(sql):
    """Corre una consulta SQL en BigQuery y devuelve un DataFrame de pandas.

    Usa la lectura rápida de BigQuery (Storage Read API). Si no está
    disponible (por ejemplo, porque la cuenta no tiene el rol «BigQuery
    Read Session User»), sigue con la lectura normal, que da el mismo
    resultado pero tarda más en tablas grandes.
    """
    trabajo = client.query(sql)
    trabajo.result()                      # si la consulta tiene un error, sale aquí
    resultado = None
    if _LECTURA_RAPIDA[0]:
        try:
            resultado = trabajo.to_arrow()
        except Exception as e:
            _LECTURA_RAPIDA[0] = False
            print("Aviso: la lectura rápida de BigQuery no está disponible "
                  f"({type(e).__name__}); se usa la lectura normal.")
    if resultado is None:
        resultado = trabajo.to_arrow(create_bqstorage_client=False)
    return resultado.to_pandas(date_as_object=False, coerce_temporal_nanoseconds=True)


def leer_tabla(tabla, columnas=None, dataset=None, filtro=None):
    """Lee una tabla de BigQuery a pandas.

    columnas  lista de columnas a traer; por defecto, todas.
    filtro    condición SQL opcional, p. ej. "anio >= 2021".

    Si la tabla se guardó con guardar_tabla(), devuelve las filas en el
    mismo orden en que se guardaron, las columnas con su nombre original
    (tildes, espacios) y el índice.
    """
    destino = nombre_completo(tabla, dataset)
    esquema = client.get_table(destino).schema
    con_orden = any(c.name == "orden_fila" for c in esquema)
    seleccion = ", ".join(f"`{c}`" for c in columnas) if columnas else "*"
    df = leer_sql(f"SELECT {seleccion} FROM `{destino}`"
                  + (f" WHERE {filtro}" if filtro else "")
                  + (" ORDER BY orden_fila" if con_orden else ""))
    df = df.drop(columns=["orden_fila"], errors="ignore")

    originales, indice, sin_nombre = {}, [], set()
    for campo in esquema:
        marca = _MARCA.match(campo.description or "")
        if marca.group(2):
            originales[campo.name] = marca.group(2)
        if marca.group(1) and campo.name in df.columns:
            indice.append(campo.name)
            if "sin nombre" in marca.group(1):
                sin_nombre.add(campo.name)
    if indice:
        df = df.set_index(indice)
        df.index = df.index.set_names(
            [None if n in sin_nombre else originales.get(n, n) for n in indice])
    return df.rename(columns=originales)


# --- 3.5 Escribir -----------------------------------------------------------
_DATASETS_LISTOS = set()


def _nombre_para_bigquery(nombre, usados):
    """Nombre de columna que BigQuery acepta siempre: letras sin tilde,
    números y guion bajo. Los que ya cumplen se dejan igual."""
    texto = str(nombre)
    if re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]{0,299}", texto) and texto.lower() not in usados:
        usados.add(texto.lower())
        return texto
    limpio = unicodedata.normalize("NFKD", texto).encode("ascii", "ignore").decode()
    limpio = re.sub(r"[^0-9A-Za-z]+", "_", limpio).strip("_").lower() or "columna"
    if limpio[0].isdigit():
        limpio = "c_" + limpio
    limpio, base, k = limpio[:280], limpio[:280], 2
    while limpio.lower() in usados:
        limpio, k = f"{base}_{k}", k + 1
    usados.add(limpio.lower())
    return limpio


def _tipos_para_bigquery(df):
    """Deja cada columna en un tipo que BigQuery reciba sin error:
    categorías a texto, enteros con vacíos a entero, mezclas a texto."""
    for c in df.columns:
        s = df[c]
        if isinstance(s.dtype, pd.CategoricalDtype):
            s = s.astype(object).where(s.notna(), None)
        if s.dtype == object:
            valores = s.dropna()
            tipos = {type(v) for v in valores}
            if not tipos or tipos <= {str}:
                pass
            elif all(isinstance(v, (bool, np.bool_)) for v in valores):
                s = s.astype("boolean")
            elif all(isinstance(v, (int, np.integer)) and not isinstance(v, bool) for v in valores):
                s = s.astype("Int64")
            elif all(isinstance(v, (int, float, np.integer, np.floating)) for v in valores):
                s = s.astype(float)
            else:
                s = s.map(lambda v: None if v is None or v is pd.NA
                          or (isinstance(v, float) and v != v) else str(v))
        df[c] = s
    return df


def guardar_tabla(df, tabla, descripcion, dataset=None, indice=False,
                  descripciones=None):
    """Escribe `df` en BigQuery. Si la tabla ya existe, la REEMPLAZA entera.

    tabla          nombre corto, p. ej. "deforestacion_ideam_celda_5km_periodo".
    descripcion    texto que se ve en BigQuery, en los detalles de la tabla.
    dataset        por defecto, DATASET_DEFORESTACION. Si no existe, se crea.
    indice         True para guardar también el índice de pandas como
                   columna(s); leer_tabla() lo recupera.
    descripciones  {columna: texto}, para documentar columnas en BigQuery.
    """
    dataset = dataset or globals().get("DATASET_DEFORESTACION")
    destino = nombre_completo(tabla, dataset)
    if dataset not in _DATASETS_LISTOS:
        ds = bigquery.Dataset(f"{PROYECTO}.{dataset}")
        ds.location = globals().get("UBICACION", "US")
        client.create_dataset(ds, exists_ok=True)
        _DATASETS_LISTOS.add(dataset)

    datos = df.copy()
    marcas = {}
    if indice:
        nombres_indice = list(datos.index.names)
        datos = datos.reset_index()
        for k, n in enumerate(nombres_indice):
            columna = datos.columns[k]
            marcas[columna] = "[índice sin nombre] " if n is None else "[índice] "
    else:
        datos = datos.reset_index(drop=True)
    if isinstance(datos.columns, pd.MultiIndex):
        datos.columns = [" | ".join(str(x) for x in c if str(x) != "") for c in datos.columns]

    descripciones = descripciones or {}
    usados, notas, columnas = set(), {}, []
    for c in datos.columns:
        nuevo = _nombre_para_bigquery(c, usados)
        texto = marcas.get(c, "")
        if nuevo != str(c):
            texto += f"Nombre original: «{c}»"
        if descripciones.get(c):
            texto += (". " if texto and not texto.endswith("] ") else "") + descripciones[c]
        notas[nuevo] = texto[:1024]
        columnas.append(nuevo)
    datos.columns = columnas
    datos = _tipos_para_bigquery(datos)
    # BigQuery no guarda el orden de las filas, y algunos cálculos (el
    # muestreo del random forest, por ejemplo) dependen de él. Esta
    # columna lo conserva; leer_tabla() la usa y la quita.
    datos["orden_fila"] = np.arange(len(datos), dtype="int64")
    notas["orden_fila"] = ("Posición de la fila en la tabla original. Sirve para leerla "
                           "en el mismo orden: ORDER BY orden_fila.")

    trabajo = client.load_table_from_dataframe(
        datos, destino,
        job_config=bigquery.LoadJobConfig(write_disposition="WRITE_TRUNCATE"))
    trabajo.result()

    t = client.get_table(destino)
    t.description = descripcion
    t.schema = [bigquery.SchemaField(f.name, f.field_type, mode=f.mode,
                                     description=notas.get(f.name) or None,
                                     fields=f.fields) for f in t.schema]
    client.update_table(t, ["description", "schema"])
    print(f"Guardada {destino}: {t.num_rows:,} filas, {len(t.schema)} columnas")

In [ ]:
import logging
import os
import sys
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from matplotlib.patches import Patch


logging.getLogger("matplotlib").setLevel(logging.WARNING)
sns.set_theme(style="whitegrid", font_scale=1.15, palette="colorblind")
plt.rcParams.update({"figure.dpi": 110, "axes.titleweight": "bold", "axes.titlesize": 13})
pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 200)

RURALES = ["Rural", "Rural disperso"]
anual = leer_tabla("deforestacion_credito_produccion_municipio_anio")
cultivos = leer_tabla("area_cosechada_eva_municipio_cultivo_anio")

## Datos y filtros

**La pregunta.** ¿Con qué municipios y con qué variables se trabaja?

**Cómo se hace.**

- **Ruralidad.** Se usa la clasificación más reciente de la tabla de municipios del Observatorio, que es la misma de
  2023 a 2026. Queda fija: un municipio es rural, o no, en todos los años.
- **Crédito.** Solo el del eslabón de producción, en millones de pesos de cada año, sin descontar la inflación. Va
  en total y por cadena: ganadería (doble propósito, leche y carne), palma de aceite, cacao, café y arroz.
- **Cultivos.** Se usa el área **cosechada** de la EVA, no la sembrada. En los cultivos con dos semestres al año, como
  el arroz, se suman los dos.
- **Años.** 2021 a 2025, los que tienen dato del IDEAM. El índice de la sección 1 usa también 2020 cuando la fuente
  lo tiene.
- **Sin filtro de bosque.** Entran todos los municipios rurales, tengan mucho o poco bosque. Donde no hay bosque, la
  tasa de deforestación no se puede calcular y queda vacía.
- Para las correlaciones (sección 4) todo se pasa a **intensidades** (por km² o como % del área).

In [ ]:
km2 = anual.area_grilla_ha / 100
d = anual.assign(
    # Deforestacion del anio
    tasa=100 * anual.tasa_def_ideam,                                    # % del bosque perdido en el anio
    def_100km2=100 * anual.def_ideam_ha / km2,                          # ha deforestadas por 100 km2
    # Credito, ganado y cultivos del mismo anio
    cred=anual.credito_produccion_mm / km2,                             # millones de pesos por km2
    cred_gan=anual.credito_ganaderia_produccion_mm / km2,
    cred_palma=anual.credito_palma_produccion_mm / km2,
    cred_cacao=anual.credito_cacao_produccion_mm / km2,
    cred_cafe=anual.credito_cafe_produccion_mm / km2,
    cred_arroz=anual.credito_arroz_produccion_mm / km2,
    bovinos=anual.bovinos / km2,                                        # animales por km2
    fincas=100 * anual.fincas_bovinas / km2,                            # fincas por 100 km2
    agricola=100 * anual.area_cosechada_ha / anual.area_grilla_ha,      # % del area del municipio
    palma=100 * anual.cosechada_palma_ha / anual.area_grilla_ha,
    cacao=100 * anual.cosechada_cacao_ha / anual.area_grilla_ha,
    cafe=100 * anual.cosechada_cafe_ha / anual.area_grilla_ha,
    arroz=100 * anual.cosechada_arroz_ha / anual.area_grilla_ha,
    # Caracteristicas fijas del municipio
    bosque=100 * anual.bosque_ideam_inicial_ha / anual.area_grilla_ha,  # % del municipio con bosque en 2021
    area=km2,
).replace([np.inf, -np.inf], np.nan)

VARIABLES = ["cred", "cred_gan", "cred_palma", "cred_cacao", "cred_cafe", "cred_arroz", "bovinos", "fincas",
             "agricola", "palma", "cacao", "cafe", "arroz", "bosque", "area"]
NOMBRES = {
    "tasa": "Deforestación IDEAM (% del bosque)", "def_100km2": "Deforestación IDEAM (ha / 100 km²)",
    "cred": "Crédito de producción (M$ / km²)", "cred_gan": "Crédito ganadero de producción (M$ / km²)",
    "cred_palma": "Crédito de producción a palma (M$ / km²)", "cred_cacao": "Crédito de producción a cacao (M$ / km²)",
    "cred_cafe": "Crédito de producción a café (M$ / km²)", "cred_arroz": "Crédito de producción a arroz (M$ / km²)",
    "bovinos": "Bovinos por km²", "fincas": "Fincas ganaderas por 100 km²",
    "agricola": "Área cosechada (% del municipio)", "palma": "Palma (% del municipio)",
    "cacao": "Cacao (% del municipio)", "cafe": "Café (% del municipio)", "arroz": "Arroz (% del municipio)",
    "bosque": "Bosque (% del municipio)", "area": "Área del municipio (km²)",
}

rural = d.ruralidad.isin(RURALES)
muestra = d[rural & d.anio.between(2021, 2025)].copy()
# Para sumar cabezas, hectareas y pesos se usa la tabla original: en d, "bovinos" ya es por km2
crudo = anual[anual.ruralidad.isin(RURALES)]
# Una fila por municipio: el promedio de 2021-2025 (y el total, para las hectareas)
municipios = (muestra.groupby("cod_dane")
              .agg(**{c: (c, "mean") for c in ["tasa", "def_100km2"] + VARIABLES},
                   def_total_ha=("def_ideam_ha", lambda s: s.sum(min_count=1)),
                   bosque_ha=("bosque_ideam_inicial_ha", "first"),
                   nombre=("nombre_municipio", "first"), departamento=("nombre_departamento", "first"),
                   ruralidad=("ruralidad", "first"))
              .astype({"nombre": str, "departamento": str, "ruralidad": str}))

print(f"Municipios en el panel: {anual.cod_dane.nunique():,}")
print(f"Rurales y rurales dispersos: {len(municipios):,}  {municipios.ruralidad.value_counts().to_dict()}")
print(f"  sin bosque en 2021 (tasa vacía): {(municipios.bosque_ha.fillna(0) == 0).sum()}")
print(f"  con menos de 1 000 ha de bosque: {(municipios.bosque_ha.fillna(0) < 1000).sum()}")

# Que parte del pais queda en los municipios rurales, 2021-2025
periodo = anual[anual.anio.between(2021, 2025)]
parte = pd.DataFrame({
    "en los rurales": periodo[periodo.ruralidad.isin(RURALES)].agg({"def_ideam_ha": "sum", "credito_produccion_mm": "sum",
                                          "bovinos": "sum", "area_cosechada_ha": "sum"}),
    "en todo el país": periodo.agg({"def_ideam_ha": "sum", "credito_produccion_mm": "sum",
                                    "bovinos": "sum", "area_cosechada_ha": "sum"})})
parte["% en los rurales"] = 100 * parte["en los rurales"] / parte["en todo el país"]
parte.index = ["Deforestación IDEAM (ha)", "Crédito de producción (M$)", "Bovinos (cabezas, suma de los años)",
               "Área cosechada (ha, 2021-2024)"]
display(parte.round(1))
rurales = periodo[periodo.ruralidad.isin(RURALES)]
print("El crédito de producción es el "
      f"{100 * rurales.credito_produccion_mm.sum() / rurales.credito_mm.sum():.0f} % "
      "del crédito de FINAGRO en los municipios rurales.")

pd.DataFrame({"variable": [NOMBRES[c] for c in ["tasa", "def_100km2"] + VARIABLES],
              "fuente": ["IDEAM"] * 2 + ["FINAGRO"] * 6 + ["ICA"] * 2 + ["EVA"] * 5 + ["IDEAM", "Grilla de 5 km"],
              "años": ["2021-2025"] * 10 + ["2021-2024"] * 5 + ["2021 (fijo)", "fijo"]}).set_index("variable")

## 1. Cómo se movieron la deforestación, el crédito, el ganado y los cultivos

**La pregunta.** Año a año, ¿la deforestación, el crédito de producción, el ganado y los cultivos suben o bajan?
¿Se mueven juntos?

**Cómo se hace.** Se suma cada variable sobre todos los municipios rurales, año por año, y se divide por su valor de
2021: es un **índice con 2021 = 100**. Cada variable va con su propio año. El IDEAM empieza en 2021 y la EVA llega
hasta 2024.

**Cómo leer el gráfico.** Un valor de 120 quiere decir 20 % más que en 2021. Cada panel muestra una variable (en
color) junto a la deforestación del IDEAM (en gris, la misma en todos los paneles), para comparar las dos curvas. La
tabla trae el índice de cada año y, en la última columna, el valor de 2021.

In [ ]:
SERIES = {"def_ideam_ha": "Deforestación IDEAM", "alertas_gfw_ha": "Alertas GFW",
          "credito_produccion_mm": "Crédito de producción", "credito_ganaderia_produccion_mm": "Crédito ganadero de producción",
          "bovinos": "Bovinos", "fincas_bovinas": "Fincas ganaderas", "area_cosechada_ha": "Área cosechada",
          "cosechada_palma_ha": "Palma", "cosechada_cacao_ha": "Cacao", "cosechada_cafe_ha": "Café",
          "cosechada_arroz_ha": "Arroz"}
UNIDADES = {"def_ideam_ha": "ha", "alertas_gfw_ha": "ha", "credito_produccion_mm": "M$",
            "credito_ganaderia_produccion_mm": "M$", "bovinos": "cabezas", "fincas_bovinas": "fincas",
            "area_cosechada_ha": "ha", "cosechada_palma_ha": "ha", "cosechada_cacao_ha": "ha",
            "cosechada_cafe_ha": "ha", "cosechada_arroz_ha": "ha"}
total = crudo.groupby("anio")[list(SERIES)].sum(min_count=1)
total.loc[2020, "def_ideam_ha"] = np.nan                           # el IDEAM no tiene 2020
EVA = ["area_cosechada_ha", "cosechada_palma_ha", "cosechada_cacao_ha", "cosechada_cafe_ha", "cosechada_arroz_ha"]
total.loc[2025, EVA] = np.nan                                      # la EVA no tiene 2025
indice = 100 * total / total.loc[2021]

fig, axes = plt.subplots(5, 2, figsize=(14, 21), sharex=True, sharey=True)
for ax, c in zip(axes.flat, [c for c in SERIES if c != "def_ideam_ha"]):
    ax.plot(indice.index, indice.def_ideam_ha, color="grey", lw=2, ls="--", marker="o", ms=5,
            label="Deforestación IDEAM")
    ax.plot(indice.index, indice[c], color="#2a78d6", lw=2.5, marker="o", ms=6, label=SERIES[c])
    ax.axhline(100, color="black", lw=0.6)
    ax.set_title(SERIES[c])
for ax in axes[:, 0]:
    ax.set_ylabel("índice (2021 = 100)")
axes.flat[0].legend(["Deforestación IDEAM", "la variable"], fontsize=11, loc="lower left")
fig.suptitle("Municipios rurales: cada variable frente a la deforestación (índice 2021 = 100)", weight="bold",
             fontsize=15)
plt.tight_layout(rect=(0, 0, 1, .98))
plt.show()

tabla = indice.T.round(0)
tabla.index = [SERIES[c] for c in tabla.index]
tabla["valor 2021"] = [f"{total.loc[2021, c]:,.0f} {UNIDADES[c]}" for c in SERIES]
tabla

## 2. ¿En cuántos municipios se concentra cada cosa? Lorenz y Pareto 80-20

**La pregunta.** ¿La deforestación está repartida en muchos municipios o concentrada en pocos? ¿Y el crédito, el
ganado y los cultivos?

**Cómo se hace.** Para cada municipio rural se toma el **total** 2021–2025 de la deforestación (ha) y del crédito de
producción (M$), y el **promedio anual** de bovinos y de área cosechada (2021–2024 para la EVA). Se usan totales, no
intensidades, porque la pregunta es dónde está la mayor parte de cada cosa. Con eso se calculan:

- La **curva de Lorenz** y el **coeficiente de Gini**, que va de 0 (todos los municipios aportan lo mismo) a 1 (un
  solo municipio tiene todo).
- La regla del **80-20**: ¿el 20 % de los municipios suma el 80 % del total? La tabla lo dice para las cuatro
  variables, y el **gráfico de Pareto** lo muestra para la deforestación, con el nombre de cada municipio.

**Cómo leer la curva de Lorenz.** Los municipios se ordenan de mayor a menor. El eje horizontal dice qué porcentaje
de municipios se lleva, y el vertical qué porcentaje del total suman. Si todos aportaran lo mismo, la curva sería la
diagonal; mientras más se despega hacia arriba, más concentrado está.

**Cómo leer el gráfico de Pareto.** Cada barra es un municipio, de mayor a menor, con lo que aporta a la
deforestación rural. La línea roja es el acumulado y la punteada, el 80 %. El gráfico llega hasta el municipio con el
que se pasa el 80 %.

In [ ]:
por_municipio = crudo[crudo.anio.between(2021, 2025)].groupby("cod_dane").agg(
    def_ha=("def_ideam_ha", lambda s: s.sum(min_count=1)),
    credito=("credito_produccion_mm", "sum"),
    bovinos=("bovinos", "mean"),
    cosechada=("area_cosechada_ha", "mean"))
CONCENTRACION = {"def_ha": "Deforestación IDEAM", "credito": "Crédito de producción", "bovinos": "Bovinos",
                 "cosechada": "Área cosechada"}


def gini(v):
    v = np.sort(np.asarray(v, float))
    n = len(v)
    return (2 * np.arange(1, n + 1) - n - 1) @ v / (n * v.sum())


fig, ax = plt.subplots(figsize=(11.5, 8.5))
filas = []
for c, nombre in CONCENTRACION.items():
    v = np.sort(por_municipio[c].dropna().clip(lower=0).values)[::-1]
    acumulado = np.r_[0, np.cumsum(v) / v.sum()]
    ax.plot(np.linspace(0, 100, len(acumulado)), 100 * acumulado, label=nombre)
    n80 = int(np.argmax(acumulado >= .8))
    filas.append({"variable": nombre, "municipios": len(v), "Gini": round(gini(v), 2),
                  "municipios que suman el 50 %": int(np.argmax(acumulado >= .5)),
                  "municipios que suman el 80 %": n80,
                  "% de municipios que suman el 80 %": round(100 * n80 / len(v), 1),
                  "% del total en el 20 % con más": round(100 * acumulado[int(round(.2 * len(v)))], 1)})
ax.plot([0, 100], [0, 100], color="grey", lw=0.8, ls="--", label="reparto igual")
ax.set(title="Curva de Lorenz, municipios rurales", xlabel="% de municipios, de mayor a menor",
       ylabel="% acumulado del total")
ax.legend()
plt.show()
concentracion = pd.DataFrame(filas).set_index("variable")
concentracion

In [ ]:
# Pareto clasico de la deforestacion: los municipios que, de mayor a menor, llegan al 80 % del total
v = por_municipio.def_ha.dropna().sort_values(ascending=False)
parte = 100 * v / v.sum()
acumulado = parte.cumsum()
n80 = int((acumulado < 80).sum()) + 1
n20 = int(round(.2 * len(v)))
p = parte.head(n80)

fig, ax = plt.subplots(figsize=(18, 9))
ax.bar(range(n80), p.values, color="#2a78d6", linewidth=0)
ax.set_xticks(range(n80), [f"{municipios.nombre[c]} ({municipios.departamento[c]})" for c in p.index],
              rotation=90, fontsize=10)
ax.set(xlim=(-.7, n80 - .3), ylabel="% de la deforestación rural 2021-2025, cada municipio")
ax.grid(False)
ax2 = ax.twinx()
ax2.plot(range(n80), acumulado.head(n80).values, color="#d62728", lw=2, marker="o", ms=3)
ax2.axhline(80, color="black", lw=0.8, ls="--")
ax2.set(ylim=(0, 100), ylabel="% acumulado")
ax2.text(n80 - 1, 76, "80 %", ha="right", va="top", fontsize=11)
ax.set_title(f"Pareto de la deforestación: {n80} de los {len(v)} municipios rurales "
             f"({100 * n80 / len(v):.0f} %) suman el 80 %")
plt.show()
print(f"El 20 % de los municipios ({n20}) suma el {acumulado.iloc[n20 - 1]:.1f} % de la deforestación rural.")
print(f"Los {n80} municipios del 80 % están en: "
      + ", ".join(f"{k} ({n})" for k, n in municipios.loc[p.index, "departamento"].value_counts().items()))

## 3. Los 10 municipios que más deforestan: ¿qué cultivos y cuánta ganadería tienen?

**La pregunta.** En los municipios donde más se deforesta, ¿qué se cultiva y cuánto ganado hay?

**Cómo se hace.**

- Se toman los 10 municipios rurales con más hectáreas deforestadas en 2021–2025, según el IDEAM.
- Para cada uno se mira el área cosechada de cada cultivo según la EVA, en promedio de **2021 a 2024** (la EVA no
  tiene 2025). Un año sin registro de un cultivo cuenta como cero.
- Los cultivos se agrupan con los grupos de la EVA. Por ejemplo, el cacao, el café y la caña están en "cultivos
  tropicales tradicionales", y el plátano, en "frutales".

**Cómo leer los resultados.**

- La **tabla** compara, para cada municipio, lo deforestado con lo cosechado.
- La **barra de colores** muestra qué parte del área cosechada va a cada grupo de cultivos.
- Los **paneles** muestran los seis cultivos con más área de cada municipio.

In [ ]:
top10 = municipios.sort_values("def_total_ha", ascending=False).head(10)
c10 = cultivos[cultivos.cod_dane.isin(top10.index) & cultivos.anio.between(2021, 2024)]
por_cultivo = (c10.groupby(["cod_dane", "grupo_cultivo", "cultivo"]).cosechada_ha.sum().div(4)
               .loc[lambda s: s > 0].rename("ha_anio").reset_index())
cosechada = por_cultivo.groupby("cod_dane").ha_anio.sum()
principal = por_cultivo.sort_values("ha_anio", ascending=False).groupby("cod_dane").first()
por_anio = c10.groupby(["cod_dane", "anio"]).cosechada_ha.sum().unstack()

ETIQUETA = (top10.nombre + " (" + top10.departamento + ")").to_dict()
resumen = pd.DataFrame({
    "ruralidad": top10.ruralidad,
    "ha deforestadas 2021-2025": top10.def_total_ha,
    "% de la deforestación rural": 100 * top10.def_total_ha / municipios.def_total_ha.sum(),
    "área cosechada (ha al año)": cosechada,
    "cultivos": por_cultivo.groupby("cod_dane").size(),
    "cultivo principal": principal.cultivo,
    "% del área cosechada": 100 * principal.ha_anio / cosechada,
    "cambio del área cosechada 2021-2024 (%)": 100 * (por_anio[2024] / por_anio[2021] - 1),
}, index=top10.index).rename(index=ETIQUETA).rename_axis("municipio")
print(f"Los 10 suman el {resumen['% de la deforestación rural'].sum():.0f} % de la deforestación rural 2021-2025")
resumen.round(1)

In [ ]:
CORTO = {"Cultivos tropicales tradicionales": "Tropicales tradicionales",
         "Cultivos para condimentos, bebidas medicinales y aromáticas": "Condimentos y aromáticas"}
por_cultivo["grupo"] = por_cultivo.grupo_cultivo.replace(CORTO)
GRUPOS = por_cultivo.groupby("grupo").ha_anio.sum().sort_values(ascending=False).index
COLOR_GRUPO = dict(zip(GRUPOS, sns.color_palette("colorblind", len(GRUPOS))))

por_grupo = por_cultivo.pivot_table(index="cod_dane", columns="grupo", values="ha_anio", aggfunc="sum",
                                    fill_value=0)[GRUPOS].loc[top10.index[::-1]]
porcentaje = 100 * por_grupo.div(por_grupo.sum(axis=1), axis=0)
fig, ax = plt.subplots(figsize=(15, 9))
porcentaje.rename(index=ETIQUETA).plot.barh(stacked=True, color=[COLOR_GRUPO[g] for g in GRUPOS], width=.75, ax=ax)
for i, cod in enumerate(porcentaje.index):
    ax.text(101, i, f"{cosechada[cod]:,.0f} ha", va="center", fontsize=11)
ax.set(xlim=(0, 112), xlabel="% del área cosechada (promedio 2021-2024)", ylabel="",
       title="Grupos de cultivos en los 10 municipios que más deforestan (de arriba abajo, de más a menos)")
ax.legend(ncol=4, fontsize=11, loc="upper center", bbox_to_anchor=(.45, -.09))
plt.show()

print("Qué cultivos hay en cada grupo, en estos 10 municipios (ha al año, sumando los 10):")
(por_cultivo.groupby(["grupo", "cultivo"]).ha_anio.sum().round(0).sort_values(ascending=False)
 .groupby(level="grupo", group_keys=False).head(4).to_frame("ha al año").loc[GRUPOS])

In [ ]:
fig, axes = plt.subplots(5, 2, figsize=(15, 25))
for ax, cod in zip(axes.flat, top10.index):
    p = por_cultivo[por_cultivo.cod_dane == cod].nlargest(6, "ha_anio").iloc[::-1]
    ax.barh(p.cultivo, p.ha_anio, color=[COLOR_GRUPO[g] for g in p.grupo])
    ax.set_title(f"{top10.nombre[cod]} ({top10.departamento[cod]})\n{cosechada[cod]:,.0f} ha cosechadas al año",
                 fontsize=13)
    ax.tick_params(labelsize=11)
    ax.set_xlabel("ha al año (2021-2024)", fontsize=11)
fig.legend(handles=[Patch(color=COLOR_GRUPO[g], label=g) for g in GRUPOS], ncol=4, fontsize=12,
           loc="lower center", bbox_to_anchor=(.5, 0))
plt.tight_layout(rect=(0, .04, 1, 1))
plt.show()

### La ganadería en los 10 municipios

**Cómo se hace.** Para cada municipio se toman, en promedio de 2021 a 2025, los bovinos y las fincas con bovinos del
ICA y el crédito ganadero de producción de FINAGRO. Se comparan con el municipio rural típico: la mediana de los 642
municipios rurales.

**Cómo leer los resultados.**

- La **tabla** trae, para cada municipio, cuántos bovinos tiene y qué parte de los bovinos de todos los municipios
  rurales son, cuántos hay por km², cuántas fincas con bovinos y cuántos bovinos por finca, y el crédito ganadero.
  La última fila es el municipio rural típico.
- El **gráfico** muestra los bovinos de cada municipio (izquierda) y los bovinos por km² (derecha). La línea roja es
  el municipio rural típico.

In [ ]:
# Ganaderia de cada municipio rural, promedio 2021-2025
anios = crudo[crudo.anio.between(2021, 2025)]
g = anios.groupby("cod_dane").agg(bovinos=("bovinos", "mean"), fincas=("fincas_bovinas", "mean"),
                                  credito=("credito_ganaderia_produccion_mm", "mean"),
                                  km2=("area_grilla_ha", "first"))
g["km2"] = g.km2 / 100
ganaderia = pd.DataFrame({
    "bovinos (cabezas)": g.bovinos,
    "% de los bovinos rurales": 100 * g.bovinos / g.bovinos.sum(),
    "bovinos por km²": g.bovinos / g.km2,
    "fincas con bovinos": g.fincas,
    "bovinos por finca": g.bovinos / g.fincas,
    "crédito ganadero de producción (M$ al año)": g.credito,
    "crédito ganadero por km² (M$)": g.credito / g.km2,
})
tipico = ganaderia.median().rename("Municipio rural típico (mediana)")
tipico["% de los bovinos rurales"] = np.nan
print(f"Los 10 tienen el {ganaderia.loc[top10.index, '% de los bovinos rurales'].sum():.1f} % "
      "de los bovinos de los municipios rurales.")
pd.concat([ganaderia.loc[top10.index].rename(index=ETIQUETA), tipico.to_frame().T]).round(1)

In [ ]:
orden = top10.index[::-1]
etiquetas = [ETIQUETA[c] for c in orden]
fig, (a1, a2) = plt.subplots(1, 2, figsize=(17, 8.5), sharey=True)
barras = a1.barh(etiquetas, ganaderia.loc[orden, "bovinos (cabezas)"] / 1000, color="#2a78d6")
a1.bar_label(barras, fmt="{:,.0f} mil", padding=4, fontsize=11)
a1.set(xlabel="miles de bovinos (promedio 2021-2025)", title="Bovinos")
a1.set_xlim(0, 1.18 * ganaderia.loc[orden, "bovinos (cabezas)"].max() / 1000)
barras = a2.barh(etiquetas, ganaderia.loc[orden, "bovinos por km²"], color="#2a78d6")
a2.bar_label(barras, fmt="{:,.0f}", padding=4, fontsize=11)
a2.axvline(tipico["bovinos por km²"], color="#d62728", ls="--", lw=2,
           label=f"municipio rural típico ({tipico['bovinos por km²']:.0f} por km²)")
a2.set(xlabel="bovinos por km² (promedio 2021-2025)", title="Bovinos por km²")
a2.set_xlim(0, 1.15 * max(ganaderia.loc[orden, "bovinos por km²"].max(), tipico["bovinos por km²"]))
a2.legend(loc="lower right", fontsize=11)
fig.suptitle("Ganadería en los 10 municipios que más deforestan (de arriba abajo, de más a menos)", weight="bold",
             fontsize=14)
plt.tight_layout()
plt.show()

## 4. Correlaciones en los municipios rurales

**La pregunta.** Entre los municipios rurales, ¿los que tienen más crédito de producción, más ganado o más cultivos
son los que más deforestan? ¿Cómo se relacionan esas variables entre sí?

**Cómo se hace.**

- **Una fila por municipio**, con el promedio de sus cinco años (2021–2025). Las cinco filas de un municipio se
  parecen mucho; juntarlas sería contar cinco veces el mismo municipio.
- Todo en **intensidades** (por km² o como % del área): en totales, un municipio grande tiene más de todo solo por
  ser grande.
- **Correlación de Spearman**, que compara el orden de los municipios y no sus valores. Así unos pocos municipios
  extremos no dominan el resultado.
- **Dos medidas de deforestación.** La **tasa** (% del bosque que se pierde al año) mide la presión sobre el bosque
  que hay; no existe donde no hay bosque. Las **hectáreas por 100 km²** miden cuánto se tumba por unidad de
  territorio, y dependen mucho de cuánto bosque queda.
- **El tamaño del municipio.** La sección 4.3 repite las correlaciones dentro de grupos de municipios de tamaño
  parecido.

### Cómo leer una correlación

- **Qué mide.** Va de −1 a 1. Positiva: cuando una variable es alta, la otra también tiende a serlo. Negativa: cuando
  una es alta, la otra tiende a ser baja. Cero: no hay relación.
- **Qué tan grande es.** Una guía usual para el valor sin signo:

  | Valor | Se lee como |
  |---|---|
  | menos de 0,1 | nada |
  | 0,1 a 0,3 | débil |
  | 0,3 a 0,5 | moderada |
  | más de 0,5 | fuerte |

- **¿Puede ser casualidad?** Se usa un **intervalo de confianza del 95 % por bootstrap**: se repite el cálculo 1 000
  veces, cada vez con una muestra de municipios escogidos al azar, y se toma el rango donde caen 95 de cada 100
  resultados. Si el intervalo cruza el cero, no se puede afirmar que haya relación.

### 4.1 Cada variable frente a la deforestación

**Cómo leer el gráfico.** Cada punto es la correlación con la deforestación y la línea, su intervalo. En azul, con
la tasa; en naranja, con las hectáreas por 100 km².

In [ ]:
MEDIDAS = {"tasa": "tasa (% del bosque)", "def_100km2": "ha / 100 km²"}
rng = np.random.default_rng(0)
tablas = []
for medida in MEDIDAS:
    datos = municipios[[medida] + VARIABLES]
    base = datos.corr(method="spearman")[medida][VARIABLES]
    replicas = pd.DataFrame([datos.iloc[rng.integers(0, len(datos), len(datos))]
                             .corr(method="spearman")[medida][VARIABLES] for _ in range(1000)])
    tablas.append(pd.DataFrame({"Spearman": base, "desde": replicas.quantile(.025),
                                "hasta": replicas.quantile(.975)}))
correlaciones = pd.concat(tablas, axis=1, keys=list(MEDIDAS.values())).sort_values(("tasa (% del bosque)", "Spearman"))

fig, ax = plt.subplots(figsize=(13, 11))
for k, (medida, color) in enumerate(zip(MEDIDAS.values(), ["#2a78d6", "#eb6834"])):
    t = correlaciones[medida]
    y = np.arange(len(t)) + (0.15 if k == 0 else -0.15)
    ax.errorbar(t.Spearman, y, xerr=[t.Spearman - t.desde, t.hasta - t.Spearman], fmt="o", color=color,
                capsize=3, label=medida)
ax.set_yticks(range(len(correlaciones)), [NOMBRES[c] for c in correlaciones.index])
ax.axvline(0, color="grey", lw=0.8)
ax.axvspan(-0.1, 0.1, color="grey", alpha=.12)
ax.legend(title="deforestación medida como", loc="lower right", fontsize=11)
ax.set(title="Correlación con la deforestación, municipios rurales",
       xlabel="Spearman (intervalo del 95 %, bootstrap)")
plt.show()
print(f"Municipios con tasa: {municipios.tasa.notna().sum()}; con ha / 100 km²: {municipios.def_100km2.notna().sum()}")
correlaciones.rename(index=NOMBRES).round(2)

### 4.2 Todas las variables entre sí

**La pregunta.** ¿Qué variables van juntas? Si dos variables están muy correlacionadas entre sí, es difícil saber
cuál de las dos es la que se relaciona con la deforestación.

**Cómo se hace.** La matriz de correlaciones de Spearman entre todas las variables. Solo se muestra la mitad de abajo,
porque la de arriba es igual. Las variables van agrupadas: deforestación, crédito, ganado, cultivos y territorio.

**Cómo leer el gráfico.** Rojo es positiva y azul, negativa; mientras más intenso el color, más fuerte. Las líneas
negras separan los grupos de variables.

In [ ]:
ORDEN_MATRIZ = ["tasa", "def_100km2", "cred", "cred_gan", "cred_palma", "cred_cacao", "cred_cafe", "cred_arroz",
                "bovinos", "fincas", "agricola", "cafe", "cacao", "palma", "arroz", "bosque", "area"]
CORTES = [2, 8, 10, 15]   # despues de: deforestacion | credito | ganado | cultivos | territorio
matriz = municipios[ORDEN_MATRIZ].corr(method="spearman")
# Sin la primera fila ni la ultima columna, que en la mitad de abajo quedarian vacias
mitad = matriz.iloc[1:, :-1]
fig, ax = plt.subplots(figsize=(20, 17))
sns.heatmap(mitad.rename(index=NOMBRES, columns=NOMBRES), mask=np.triu(np.ones_like(mitad, bool), k=1),
            annot=True, fmt=".2f", cmap="RdBu_r", vmin=-1, vmax=1, annot_kws={"size": 10}, ax=ax)
ax.grid(False)
for corte in CORTES:
    ax.hlines(corte - 1, 0, corte, color="black", lw=1.2)
    ax.vlines(corte, corte - 1, len(mitad), color="black", lw=1.2)
ax.set_title("Correlaciones de Spearman entre todas las variables, municipios rurales")
plt.show()

# Bootstrap de la matriz completa, para saber que pares no se distinguen de cero
rng = np.random.default_rng(1)
replicas_matriz = np.array([municipios[ORDEN_MATRIZ].iloc[rng.integers(0, len(municipios), len(municipios))]
                            .corr(method="spearman").values for _ in range(500)])
desde, hasta = np.percentile(replicas_matriz, 2.5, 0), np.percentile(replicas_matriz, 97.5, 0)
i, j = np.tril_indices(len(ORDEN_MATRIZ), -1)
pares = pd.DataFrame({"variable A": [NOMBRES[ORDEN_MATRIZ[k]] for k in i],
                      "variable B": [NOMBRES[ORDEN_MATRIZ[k]] for k in j],
                      "Spearman": matriz.values[i, j], "desde": desde[i, j], "hasta": hasta[i, j]})
pares = pares[~((pares["variable A"].str.startswith("Deforestación")) & (pares["variable B"].str.startswith("Deforestación")))]
pares["tamaño"] = pd.cut(pares.Spearman.abs(), [0, .1, .3, .5, 1], labels=["nada", "débil", "moderada", "fuerte"],
                         include_lowest=True)
print(f"Pares de variables: {len(pares)}")
print("Cuántos hay de cada tamaño:", pares["tamaño"].value_counts().reindex(["fuerte", "moderada", "débil", "nada"]).to_dict())
print(f"Pares cuyo intervalo cruza el cero: {((pares.desde < 0) & (pares.hasta > 0)).sum()}")
print("\nLos 12 pares más fuertes:")
pares.reindex(pares.Spearman.abs().sort_values(ascending=False).index).head(12).round(2).reset_index(drop=True)

### 4.3 Las correlaciones según el tamaño del municipio

**La pregunta.** ¿Las correlaciones de 4.1 se mantienen entre municipios de tamaño parecido? El tamaño del
municipio puede estar relacionado a la vez con la deforestación y con el crédito, el ganado y los cultivos por km². Si
es así, al comparar todos los municipios juntos, una parte de cada correlación vendría solo del tamaño.

**Cómo se hace.**

- Se ordenan los municipios rurales por área y se parten en **cinco grupos con casi el mismo número de municipios**
  (quintiles): muy pequeños, pequeños, medianos, grandes y muy grandes. No es una clasificación oficial: no hay una
  clasificación oficial de municipios por área. El área es la de la grilla de 5 km.
- Dentro de cada grupo se calcula la misma correlación de Spearman de 4.1, con las dos medidas de deforestación.
- **¿Cuándo cuenta una correlación?** Cada grupo tiene unos 128 municipios. Una regla aproximada: con *n*
  municipios, una correlación menor que 2/√*n* en valor absoluto no se distingue de cero. Con 128 municipios, eso es
  0,18.
- Al final se revisa que el resultado no dependa de haber usado cinco grupos: se repite con 3 y con 10.

**Cómo leer los resultados.** La primera tabla describe el municipio típico (la mediana) de cada grupo. En los mapas
de calor, cada columna es un grupo; la primera, "Todos", es la de la sección 4.1. Las celdas con color pasan el umbral
de su grupo; las celdas en blanco, con el número en gris, no lo pasan.

In [ ]:
TAMANOS = ["Muy pequeños", "Pequeños", "Medianos", "Grandes", "Muy grandes"]
municipios["tamano"] = pd.qcut(municipios.area, 5, labels=TAMANOS)

perfil = municipios.groupby("tamano", observed=True).agg(
    municipios=("area", "size"), **{"área desde (km²)": ("area", "min"), "área hasta (km²)": ("area", "max")},
    **{NOMBRES[c]: (c, "median") for c in ["tasa", "def_100km2", "cred", "bovinos", "fincas", "agricola", "bosque"]})
perfil.T.round(2)

In [ ]:
ORDEN_FILAS = list(correlaciones.index[::-1])   # de la mas positiva a la mas negativa con la tasa, como en 4.1
grupos = {"Todos": municipios, **dict(list(municipios.groupby("tamano", observed=True)))}
por_tamano = {}
fig, axes = plt.subplots(2, 1, figsize=(15, 25))
for ax, (medida, titulo) in zip(axes, MEDIDAS.items()):
    tabla = pd.DataFrame({k: g[[medida] + VARIABLES].corr(method="spearman")[medida][ORDEN_FILAS]
                          for k, g in grupos.items()})
    n = pd.Series({k: g[medida].notna().sum() for k, g in grupos.items()})
    cuenta = tabla.abs() >= 2 / np.sqrt(n)
    sns.heatmap(tabla.where(cuenta).rename(index=NOMBRES), cmap="RdBu_r", vmin=-.8, vmax=.8,
                ax=ax)
    for (fila, col), v in np.ndenumerate(tabla.values):
        ax.text(col + .5, fila + .5, f"{v:.2f}", ha="center", va="center", fontsize=12,
                color=("white" if abs(v) > .6 else "black") if cuenta.values[fila, col] else "#b0b0b0")
    ax.set_xticklabels([f"{k}\n({n[k]} mun.)" for k in tabla.columns], rotation=0, fontsize=11)
    ax.axvline(1, color="black", lw=1.2)
    ax.set(title=f"Correlación con la deforestación medida como {titulo}", xlabel="", ylabel="")
    ax.grid(False)
    por_tamano[titulo] = tabla.rename(index=NOMBRES)
plt.tight_layout()
plt.show()
pd.concat(por_tamano, axis=1).round(2)

In [ ]:
# ¿Depende de haber usado cinco grupos? Promedio de las correlaciones con la tasa dentro de los grupos, ponderado
# por el numero de municipios de cada grupo
revision = {"Todos (4.1)": correlaciones[("tasa (% del bosque)", "Spearman")]}
for k in (3, 5, 10):
    grupo = pd.qcut(municipios.area, k, labels=False)
    dentro = municipios.groupby(grupo).apply(lambda t: t[["tasa"] + VARIABLES].corr(method="spearman")["tasa"][VARIABLES])
    peso = municipios.groupby(grupo).tasa.count()
    revision[f"{k} grupos"] = dentro.mul(peso, axis=0).sum() / peso.sum()
pd.DataFrame(revision).loc[ORDEN_FILAS].rename(index=NOMBRES).round(2)

## 5. Mapa de la deforestación de todos los municipios

**La pregunta.** ¿Dónde se deforesta en Colombia?

**Cómo se hace.** Un mapa con los límites municipales del DANE (Marco Geoestadístico Nacional 2025), con **todos** los
municipios, rurales o no. Cada municipio se colorea de dos formas:

- Por las **hectáreas deforestadas** en 2021–2025, según el IDEAM.
- Por la **tasa** de deforestación: el promedio 2021–2025 del % de su bosque que pierde cada año.

San Andrés y Providencia van en un recuadro.

**Cómo leer el mapa.** Mientras más oscuro, más deforestación. En gris claro, los municipios que no perdieron nada
en los cinco años; en blanco con rayas, los que no tienen dato (en la tasa, los que no tienen bosque).

In [ ]:
# Límites de los municipios para el mapa: DANE, Marco Geoestadístico
# Nacional 2025, versión "Gráfico" (CC BY 4.0). Se descargan aquí, en
# memoria, y no se guardan en BigQuery: el Observatorio ya los tiene.
import requests
_r = requests.get(
    "https://services.arcgis.com/ioNRSZMYYlx0PUaB/arcgis/rest/services/"
    "MGN2025_MPIO_GRAFICO/FeatureServer/0/query",
    params={"where": "1=1", "outFields": "mpio_cdpmp,mpio_cnmbr", "outSR": "4326", "f": "geojson"},
    timeout=300)
_r.raise_for_status()
limites = gpd.GeoDataFrame.from_features(_r.json()["features"], crs="EPSG:4326")
# Simplificados a unos 50 m: alcanza para el mapa y se dibuja más rápido.
limites["geometry"] = limites.geometry.simplify(0.0005, preserve_topology=True)
limites = limites[["mpio_cdpmp", "mpio_cnmbr", "geometry"]]
todos = d[d.anio.between(2021, 2025)].groupby("cod_dane").agg(
    def_total_ha=("def_ideam_ha", lambda s: s.sum(min_count=1)), tasa=("tasa", "mean"))
mapa = limites.merge(todos, left_on="mpio_cdpmp", right_index=True, how="left")
print(f"Municipios en el mapa: {len(mapa):,}; sin dato de deforestación: {mapa.def_total_ha.isna().sum()}")
print(f"Del panel sin polígono en el mapa: {sorted(set(todos.index) - set(limites.mpio_cdpmp))}")

CLASES = {
    "def_total_ha": ("Hectáreas deforestadas 2021-2025", [0, 10, 100, 1_000, 10_000],
                     ["0", "hasta 10", "10 a 100", "100 a 1 000", "1 000 a 10 000", "más de 10 000"]),
    "tasa": ("Tasa de deforestación (% del bosque por año)", [0, 0.01, 0.1, 0.5, 1],
             ["0", "hasta 0,01", "0,01 a 0,1", "0,1 a 0,5", "0,5 a 1", "más de 1"]),
}
COLORES = ["#e3e3e3"] + list(sns.color_palette("YlOrRd", 5).as_hex())
ISLAS = mapa.mpio_cdpmp.str.startswith("88")

for c, (titulo, cortes, etiquetas) in CLASES.items():
    fig, ax = plt.subplots(figsize=(13, 16))
    clase = pd.cut(mapa[c], [-np.inf] + cortes + [np.inf], labels=etiquetas, right=True)
    color = clase.map(dict(zip(etiquetas, COLORES))).astype(object).fillna("white")
    for a, filtro in ((ax, ~ISLAS), (ax.inset_axes([0.0, 0.78, 0.2, 0.2]), ISLAS)):
        m = mapa[filtro]
        m.plot(ax=a, color=color[filtro], edgecolor="#9a9a9a", linewidth=0.1)
        m[clase[filtro].isna()].plot(ax=a, facecolor="none", edgecolor="#9a9a9a", hatch="////", linewidth=0.1)
        a.set_xticks([])
        a.set_yticks([])
    ax.set_title(titulo, fontsize=16)
    ax.set_axis_off()
    ax.legend(handles=[Patch(facecolor=col, edgecolor="#9a9a9a", label=e) for col, e in zip(COLORES, etiquetas)]
              + [Patch(facecolor="white", edgecolor="#9a9a9a", hatch="////", label="sin dato")],
              loc="lower left", fontsize=13, frameon=True)
    plt.tight_layout()
    plt.show()